# Financial Embedding Engine

**Project:** Financial Planning & Analysis Intelligence Platform

**Notebook:** `07-financial-embedding-engine.ipynb`

In [2]:
# ==========================================
# Notebook 07
# Financial Embedding Engine
# ==========================================

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
financial_df = pd.read_csv("../data/financial_risk_dataset.csv")

financial_df.head()

,ticker,quarter,revenue_million,gross_margin_pct,operating_income_million,net_income_million,eps,earnings_call,risk_factors,mda_section,...,character_count,sentences,chunks,sentiment,confidence,weighted_sentiment,guidance_score,positive_guidance_score,total_risk_score,normalized_risk_score
0,ABC,2023-Q1,120,58,22,16,1.20,\n Demand remained strong across enterp...,\n Inflation remains a concern.\n ...,\n Management believes demand trends re...,...,299,['Demand remained strong across enterprise cus...,['Demand remained strong across enterprise cus...,positive,0.956167,0.956167,1,1,2,1.0
1,ABC,2023-Q2,128,60,25,18,1.35,\n Enterprise adoption accelerated.\n ...,\n Competitive pressure exists.\n ...,\n New product launches contributed pos...,...,239,"['Enterprise adoption accelerated.', 'Pricing ...",['Enterprise adoption accelerated. Pricing pre...,positive,0.957692,0.957692,1,2,1,0.5
2,ABC,2023-Q3,138,61,28,21,1.52,\n Customer demand exceeded expectation...,\n Geopolitical uncertainty remains.\n ...,\n Strong customer growth across region...,...,269,"['Customer demand exceeded expectations.', 'Su...",['Customer demand exceeded expectations. Suppl...,positive,0.951876,0.951876,2,0,1,0.5
3,ABC,2023-Q4,150,63,32,24,1.72,\n Record quarter performance.\n ...,\n Macroeconomic slowdown remains possi...,\n Revenue growth exceeded internal exp...,...,260,"['Record quarter performance.', 'Management ex...",['Record quarter performance. Management expec...,positive,0.956653,0.956653,1,1,0,0.0
4,ABC,2024-Q1,158,62,34,26,1.85,\n Strong start to the fiscal year driv...,\n Increased cyber security threats req...,\n Gross margin slightly contracted due...,...,460,['Strong start to the fiscal year driven by en...,['Strong start to the fiscal year driven by en...,positive,0.922371,0.922371,1,0,0,0.0


In [4]:
print("Rows:", len(financial_df))

print("Columns:", len(financial_df.columns))

financial_df.columns

Rows: 12
Columns: 22


Index(['ticker', 'quarter', 'revenue_million', 'gross_margin_pct',
       'operating_income_million', 'net_income_million', 'eps',
       'earnings_call', 'risk_factors', 'mda_section', 'combined_text',
       'clean_text', 'character_count', 'sentences', 'chunks', 'sentiment',
       'confidence', 'weighted_sentiment', 'guidance_score',
       'positive_guidance_score', 'total_risk_score', 'normalized_risk_score'],
      dtype='object')

In [5]:
financial_df["financial_document"] = (
    financial_df["earnings_call"].fillna("")
    + " "
    + financial_df["risk_factors"].fillna("")
    + " "
    + financial_df["mda_section"].fillna("")
)

In [6]:
financial_df[["quarter", "financial_document"]]

,quarter,financial_document
0,2023-Q1,\n Demand remained strong across enterp...
1,2023-Q2,\n Enterprise adoption accelerated.\n ...
2,2023-Q3,\n Customer demand exceeded expectation...
3,2023-Q4,\n Record quarter performance.\n ...
4,2024-Q1,\n Strong start to the fiscal year driv...
5,2024-Q2,\n New AI-driven product modules saw re...
6,2024-Q3,\n European expansion is yielding solid...
7,2024-Q4,"\n An exceptional finish to 2024, cross..."
8,2025-Q1,\n We carried the strong closing moment...
9,2025-Q2,\n Our SaaS migration is officially com...


In [7]:
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [8]:
financial_embeddings = embedding_model.encode(
    financial_df["financial_document"].tolist(), show_progress_bar=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
financial_embeddings.shape

(12, 768)

In [10]:
financial_embeddings[0][:20]

array([ 0.0205192 ,  0.00533664, -0.03920348, -0.02181859,  0.04920003,
        0.00118104, -0.0261846 , -0.02176713, -0.06527634, -0.0349213 ,
       -0.04080717, -0.01249505,  0.01108264,  0.0767099 , -0.03077228,
        0.01320018, -0.02088709,  0.01638892, -0.05027859,  0.00812198],
      dtype=float32)

In [11]:
embedding_df = pd.DataFrame(financial_embeddings)

In [12]:
embedding_df.head()

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,0.020519,0.005337,-0.039203,-0.021819,0.049200,0.001181,-0.026185,-0.021767,-0.065276,-0.034921,...,0.005386,0.059427,0.100701,0.015887,-0.023659,-0.033474,-0.002497,0.001263,-0.012767,-0.046298
1,0.008946,0.012003,-0.035798,-0.057973,0.048478,-0.032285,-0.022696,0.011887,-0.086582,-0.042413,...,-0.028052,0.057678,0.045884,0.017545,-0.045172,0.001591,-0.005883,-0.022504,-0.019212,-0.009736
2,0.016183,0.035013,-0.033805,-0.062928,0.052940,-0.023623,0.020148,-0.000985,-0.071764,-0.023113,...,-0.022246,0.045896,0.029788,0.011651,-0.014661,-0.062926,-0.010493,-0.014996,-0.020954,-0.011818
3,-0.008287,0.048717,-0.013528,-0.046675,0.051818,-0.016197,-0.045419,-0.004078,-0.067201,-0.055647,...,-0.058276,0.033997,0.056670,0.044685,-0.042262,0.029104,0.004350,0.002729,0.007463,-0.033975
4,-0.043102,0.054262,-0.009761,-0.030979,0.038890,-0.032354,-0.052823,0.034513,-0.085597,-0.043436,...,-0.051459,0.011908,0.037479,0.028172,-0.011268,-0.051856,0.034030,-0.035978,0.021501,-0.019921


In [13]:
similarity_matrix = cosine_similarity(financial_embeddings)

In [14]:
pd.DataFrame(similarity_matrix).head()

,0,1,2,3,4,5,6,7,8,9,10,11
0,1.000000,0.747253,0.784428,0.740653,0.611446,0.593411,0.642400,0.607240,0.660666,0.515730,0.603108,0.418673
1,0.747253,1.000000,0.743271,0.781666,0.741562,0.713684,0.667248,0.704513,0.754918,0.608954,0.672072,0.482315
2,0.784428,0.743271,1.000000,0.730594,0.614518,0.702919,0.736075,0.663904,0.655438,0.608042,0.697187,0.474047
3,0.740653,0.781666,0.730594,1.000000,0.687229,0.632935,0.624686,0.713615,0.749439,0.544293,0.666045,0.559343
4,0.611446,0.741562,0.614518,0.687229,1.000000,0.587072,0.589300,0.658423,0.785742,0.518071,0.601713,0.421458


In [15]:
similarity_df = pd.DataFrame(
    similarity_matrix, index=financial_df["quarter"], columns=financial_df["quarter"]
)

similarity_df

quarter,2023-Q1,2023-Q2,2023-Q3,2023-Q4,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4
quarter,,,,,,,,,,,,
2023-Q1,1.000000,0.747253,0.784428,0.740653,0.611446,0.593411,0.642400,0.607240,0.660666,0.515730,0.603108,0.418673
2023-Q2,0.747253,1.000000,0.743271,0.781666,0.741562,0.713684,0.667248,0.704513,0.754918,0.608954,0.672072,0.482315
2023-Q3,0.784428,0.743271,1.000000,0.730594,0.614518,0.702919,0.736075,0.663904,0.655438,0.608042,0.697187,0.474047
2023-Q4,0.740653,0.781666,0.730594,1.000000,0.687229,0.632935,0.624686,0.713615,0.749439,0.544293,0.666045,0.559343
2024-Q1,0.611446,0.741562,0.614518,0.687229,1.000000,0.587072,0.589300,0.658423,0.785742,0.518071,0.601713,0.421458
2024-Q2,0.593411,0.713684,0.702919,0.632935,0.587072,1.000000,0.592487,0.613105,0.610648,0.594317,0.660089,0.523921
2024-Q3,0.642400,0.667248,0.736075,0.624686,0.589300,0.592487,1.000000,0.663601,0.697505,0.622479,0.642588,0.428818
2024-Q4,0.607240,0.704513,0.663904,0.713615,0.658423,0.613105,0.663601,1.000000,0.757472,0.682544,0.639363,0.599213
2025-Q1,0.660666,0.754918,0.655438,0.749439,0.785742,0.610648,0.697505,0.757472,1.000000,0.660608,0.662777,0.499541


In [16]:
query_index = 0

scores = similarity_matrix[query_index]

scores

array([1.0000001 , 0.7472525 , 0.7844285 , 0.74065256, 0.6114465 ,
       0.5934106 , 0.6424001 , 0.6072396 , 0.6606658 , 0.51573014,
       0.60310835, 0.41867304], dtype=float32)

In [17]:
ranked_idx = np.argsort(scores)[::-1]

ranked_idx

array([ 0,  2,  1,  3,  8,  6,  4,  7, 10,  5,  9, 11], dtype=int64)

In [18]:
for idx in ranked_idx:

    print("=" * 80)

    print(financial_df.iloc[idx]["quarter"])

    print(round(scores[idx], 4))

2023-Q1
1.0
2023-Q3
0.7844
2023-Q2
0.7473
2023-Q4
0.7407
2025-Q1
0.6607
2024-Q3
0.6424
2024-Q1
0.6114
2024-Q4
0.6072
2025-Q3
0.6031
2024-Q2
0.5934
2025-Q2
0.5157
2025-Q4
0.4187


In [19]:
def financial_semantic_search(query, top_k=3):

    query_embedding = embedding_model.encode(query)

    scores = cosine_similarity(query_embedding.reshape(1, -1), financial_embeddings)[0]

    ranked_idx = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in ranked_idx:

        results.append(
            {
                "quarter": financial_df.iloc[idx]["quarter"],
                "score": float(scores[idx]),
                "document": financial_df.iloc[idx]["financial_document"],
            }
        )

    return pd.DataFrame(results)

In [20]:
financial_semantic_search("strong customer demand")

,quarter,score,document
0,2023-Q1,0.725299,\n Demand remained strong across enterp...
1,2023-Q3,0.584843,\n Customer demand exceeded expectation...
2,2023-Q4,0.522575,\n Record quarter performance.\n ...


In [21]:
financial_semantic_search("supply chain challenges")

,quarter,score,document
0,2023-Q3,0.529869,\n Customer demand exceeded expectation...
1,2023-Q1,0.474401,\n Demand remained strong across enterp...
2,2024-Q3,0.347832,\n European expansion is yielding solid...


In [22]:
financial_semantic_search("inflation risk")

,quarter,score,document
0,2023-Q1,0.284839,\n Demand remained strong across enterp...
1,2025-Q1,0.219955,\n We carried the strong closing moment...
2,2024-Q3,0.214734,\n European expansion is yielding solid...


In [23]:
quarter_similarity = []

for i in range(len(financial_df)):

    for j in range(i + 1, len(financial_df)):

        quarter_similarity.append(
            {
                "quarter_1": financial_df.iloc[i]["quarter"],
                "quarter_2": financial_df.iloc[j]["quarter"],
                "similarity": similarity_matrix[i][j],
            }
        )

In [24]:
quarter_similarity_df = pd.DataFrame(quarter_similarity)

quarter_similarity_df

,quarter_1,quarter_2,similarity
0,2023-Q1,2023-Q2,0.747253
1,2023-Q1,2023-Q3,0.784428
2,2023-Q1,2023-Q4,0.740653
3,2023-Q1,2024-Q1,0.611446
4,2023-Q1,2024-Q2,0.593411
...,...,...,...
61,2025-Q1,2025-Q3,0.662777
62,2025-Q1,2025-Q4,0.499541
63,2025-Q2,2025-Q3,0.568516
64,2025-Q2,2025-Q4,0.489369


In [25]:
quarter_similarity_df.sort_values(by="similarity", ascending=False)

,quarter_1,quarter_2,similarity
41,2024-Q1,2025-Q1,0.785742
1,2023-Q1,2023-Q3,0.784428
12,2023-Q2,2023-Q4,0.781666
56,2024-Q4,2025-Q1,0.757472
17,2023-Q2,2025-Q1,0.754918
...,...,...,...
20,2023-Q2,2025-Q4,0.482315
29,2023-Q3,2025-Q4,0.474047
55,2024-Q3,2025-Q4,0.428818
44,2024-Q1,2025-Q4,0.421458


In [26]:
print("Embedding Shape:", financial_embeddings.shape)

print("Embedding Dimension:", financial_embeddings.shape[1])

Embedding Shape: (12, 768)
Embedding Dimension: 768


In [27]:
np.save("../data/financial_embeddings.npy", financial_embeddings)

In [28]:
similarity_df.to_csv("../data/financial_similarity_matrix.csv")

In [29]:
similarity_df

quarter,2023-Q1,2023-Q2,2023-Q3,2023-Q4,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3,2025-Q4
quarter,,,,,,,,,,,,
2023-Q1,1.000000,0.747253,0.784428,0.740653,0.611446,0.593411,0.642400,0.607240,0.660666,0.515730,0.603108,0.418673
2023-Q2,0.747253,1.000000,0.743271,0.781666,0.741562,0.713684,0.667248,0.704513,0.754918,0.608954,0.672072,0.482315
2023-Q3,0.784428,0.743271,1.000000,0.730594,0.614518,0.702919,0.736075,0.663904,0.655438,0.608042,0.697187,0.474047
2023-Q4,0.740653,0.781666,0.730594,1.000000,0.687229,0.632935,0.624686,0.713615,0.749439,0.544293,0.666045,0.559343
2024-Q1,0.611446,0.741562,0.614518,0.687229,1.000000,0.587072,0.589300,0.658423,0.785742,0.518071,0.601713,0.421458
2024-Q2,0.593411,0.713684,0.702919,0.632935,0.587072,1.000000,0.592487,0.613105,0.610648,0.594317,0.660089,0.523921
2024-Q3,0.642400,0.667248,0.736075,0.624686,0.589300,0.592487,1.000000,0.663601,0.697505,0.622479,0.642588,0.428818
2024-Q4,0.607240,0.704513,0.663904,0.713615,0.658423,0.613105,0.663601,1.000000,0.757472,0.682544,0.639363,0.599213
2025-Q1,0.660666,0.754918,0.655438,0.749439,0.785742,0.610648,0.697505,0.757472,1.000000,0.660608,0.662777,0.499541


In [ ]:
.to_csv("../data/financial_embedding_corpus.csv", index=False)financial_df

In [31]:
financial_df

,ticker,quarter,revenue_million,gross_margin_pct,operating_income_million,net_income_million,eps,earnings_call,risk_factors,mda_section,...,sentences,chunks,sentiment,confidence,weighted_sentiment,guidance_score,positive_guidance_score,total_risk_score,normalized_risk_score,financial_document
0,ABC,2023-Q1,120,58,22,16,1.20,\n Demand remained strong across enterp...,\n Inflation remains a concern.\n ...,\n Management believes demand trends re...,...,['Demand remained strong across enterprise cus...,['Demand remained strong across enterprise cus...,positive,0.956167,0.956167,1,1,2,1.0,\n Demand remained strong across enterp...
1,ABC,2023-Q2,128,60,25,18,1.35,\n Enterprise adoption accelerated.\n ...,\n Competitive pressure exists.\n ...,\n New product launches contributed pos...,...,"['Enterprise adoption accelerated.', 'Pricing ...",['Enterprise adoption accelerated. Pricing pre...,positive,0.957692,0.957692,1,2,1,0.5,\n Enterprise adoption accelerated.\n ...
2,ABC,2023-Q3,138,61,28,21,1.52,\n Customer demand exceeded expectation...,\n Geopolitical uncertainty remains.\n ...,\n Strong customer growth across region...,...,"['Customer demand exceeded expectations.', 'Su...",['Customer demand exceeded expectations. Suppl...,positive,0.951876,0.951876,2,0,1,0.5,\n Customer demand exceeded expectation...
3,ABC,2023-Q4,150,63,32,24,1.72,\n Record quarter performance.\n ...,\n Macroeconomic slowdown remains possi...,\n Revenue growth exceeded internal exp...,...,"['Record quarter performance.', 'Management ex...",['Record quarter performance. Management expec...,positive,0.956653,0.956653,1,1,0,0.0,\n Record quarter performance.\n ...
4,ABC,2024-Q1,158,62,34,26,1.85,\n Strong start to the fiscal year driv...,\n Increased cyber security threats req...,\n Gross margin slightly contracted due...,...,['Strong start to the fiscal year driven by en...,['Strong start to the fiscal year driven by en...,positive,0.922371,0.922371,1,0,0,0.0,\n Strong start to the fiscal year driv...
5,ABC,2024-Q2,167,64,38,29,2.05,\n New AI-driven product modules saw re...,\n Intense competition in the mid-marke...,\n Expansion of gross margin driven by ...,...,['New AIdriven product modules saw record adop...,['New AIdriven product modules saw record adop...,positive,0.921203,0.921203,2,0,2,1.0,\n New AI-driven product modules saw re...
6,ABC,2024-Q3,174,64,41,31,2.18,\n European expansion is yielding solid...,\n Evolving global data privacy laws co...,\n Operating margin expanded due to sca...,...,['European expansion is yielding solid initial...,['European expansion is yielding solid initial...,positive,0.919702,0.919702,1,0,1,0.5,\n European expansion is yielding solid...
7,ABC,2024-Q4,190,65,46,35,2.45,"\n An exceptional finish to 2024, cross...",\n Potential shifts in corporate tax ra...,\n Record annual performance driven by ...,...,"['An exceptional finish to 2024, crossing hist...","['An exceptional finish to 2024, crossing hist...",positive,0.952842,0.952842,1,2,0,0.0,"\n An exceptional finish to 2024, cross..."
8,ABC,2025-Q1,198,64,48,36,2.52,\n We carried the strong closing moment...,\n Geopolitical trade restrictions coul...,\n Growth continues at a stable pace de...,...,['We carried the strong closing momentum of la...,['We carried the strong closing momentum of la...,negative,0.740657,-0.740657,2,2,0,0.0,\n We carried the strong closing moment...
9,ABC,2025-Q2,210,66,53,40,2.80,\n Our SaaS migration is officially com...,\n Slowing down of tech spending in the...,\n Gross margins hit a record high due ...,...,"['Our SaaS migration is officially complete, d...","['Our SaaS migration is officially complete, d...",positive,0.953374,0.953374,1,0,0,0.0,\n Our SaaS migration is officially com...


In [32]:
loaded_embeddings = np.load("../data/financial_embeddings.npy")

loaded_embeddings.shape

(12, 768)